<a href="https://colab.research.google.com/github/Sridipta-Roy/protein-function-active-learning/blob/main/notebooks/05_esm_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Sridipta-Roy/protein-function-active-learning

Cloning into 'protein-function-active-learning'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 62 (delta 9), reused 57 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 16.43 MiB | 13.59 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [2]:
%cd protein-function-active-learning

/content/protein-function-active-learning


In [3]:
import os
print(os.path.exists("data/processed/labeled_dataset.csv"))

True


# 05 - ESM-2 protein language model embeddings

This notebook generates frozen, mean-pooled **ESM-2** embeddings (1280-dim) for every protein.

**Why this step:** the Day 5 baselines plateaued at macro-F1 ~0.527 with two
boosting models agreeing to within 0.002 — highlighting that the ceiling is the *feature
representation*, not the classifier. ESM embeddings are a richer representation
and are the natural next move.

**Key discipline:** we apply the **same duplicate-sequence drop** as notebook 04
so the embedding rows line up with the handcrafted-feature rows. The actual
train/test split happens in later notebooks, on whichever feature set is being
evaluated — here we only produce one embedding per (deduplicated) protein.

> Ran this on **Colab with a GPU** (Runtime -> Change runtime type -> GPU).
> The 650M model embeds ~7-8k proteins in a few minutes on a T4.


## 1. Setup

In [4]:
!pip install -q torch transformers tqdm


In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# --- Project paths -------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [8]:
# For Colab
PROJECT_ROOT = Path("/content/protein-function-active-learning")

In [9]:
DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMB_DIR       = DATA_DIR / "embeddings"
SRC_DIR       = PROJECT_ROOT / "src"

EMB_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(SRC_DIR))

In [10]:
from embeddings import ESMEmbedder, save_embeddings, load_embeddings

## 2. Load the labeled dataset and drop duplicate sequences

Same leakage control as notebook 04: identical sequences must not straddle the
train/test boundary. We dedup *here* so the saved embeddings already reflect the
modeling-ready set of proteins.

In [11]:
df = pd.read_csv(PROCESSED_DIR / "labeled_dataset.csv")
print(f"Loaded {len(df):,} proteins")

# Drop exact duplicate sequences, keep first occurrence (matches earlier logic).
before = len(df)
df = df.drop_duplicates(subset=["sequence"]).reset_index(drop=True)
print(f"Dropped {before - len(df):,} duplicate sequences -> {len(df):,} unique")

# Sanity check: required columns present
assert {"accession", "sequence"}.issubset(df.columns), "Need accession + sequence columns"
df[["accession", "sequence"]].head()


Loaded 8,520 proteins
Dropped 7 duplicate sequences -> 8,513 unique


,accession,sequence
0,A0A1B0GTW7,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...
1,A0JP26,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...
2,A0PK11,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...
3,A1A4S6,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...
4,A1A519,MKRRQKRKHLENEESQETAEKGGGMSKSQEDALQPGSTRVAKGWSQ...


## 3. Load ESM-2 and embed

`facebook/esm2_t33_650M_UR50D` -> 1280-dim.

In [12]:
embedder = ESMEmbedder(
    model_name="facebook/esm2_t33_650M_UR50D",
    max_length=1024,   # within ESM-2's 1022-residue limit (+ special tokens)
)
print(f"Model: {embedder.model_name}")
print(f"Device: {embedder.device}")
print(f"Embedding dim: {embedder.dim}")


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: facebook/esm2_t33_650M_UR50D
Device: cuda
Embedding dim: 1280


In [13]:
embeddings, accessions = embedder.embed_dataframe(
    df,
    id_col="accession",
    seq_col="sequence",
    batch_size=8,      # can be lowered to 4 or 2 if we hit CUDA OOM
)

print("Embeddings shape:", embeddings.shape)   # (n_proteins, 1280)
print("First accession:", accessions[0])


Embedding:   0%|          | 0/1065 [00:00<?, ?batch/s]

Embeddings shape: (8513, 1280)
First accession: A0A1B0GTW7


In [14]:
embeddings[0]

array([ 0.03243124, -0.10950679,  0.02343445, ..., -0.10604121,
       -0.03391429, -0.03071677], dtype=float32)

### Sanity checks

In [15]:
# No NaNs/Infs, row count matches, dim matches the model.
assert np.isfinite(embeddings).all(), "Found NaN/Inf in embeddings"
assert embeddings.shape[0] == len(df) == len(accessions), "Row count mismatch"
assert embeddings.shape[1] == embedder.dim, "Dim mismatch"

print("All checks passed.")
print(f"Mean L2 norm per vector: {np.linalg.norm(embeddings, axis=1).mean():.2f}")


All checks passed.
Mean L2 norm per vector: 7.08


## 4. Save

Stored as `.npy` + an aligned `metadata.csv` (row order = accession).
A vector database is intentionally *not* used here: the full matrix is small (~40 MB) and every
downstream step wants it in memory at once.

In [16]:
emb_path, meta_path = save_embeddings(embeddings, accessions, EMB_DIR)
print(f"Saved embeddings -> {emb_path}")
print(f"Saved metadata   -> {meta_path}")

size_mb = emb_path.stat().st_size / 1e6
print(f"Embedding file size: {size_mb:.1f} MB")


Saved embeddings -> /content/protein-function-active-learning/data/embeddings/esm_embeddings.npy
Saved metadata   -> /content/protein-function-active-learning/data/embeddings/metadata.csv
Embedding file size: 43.6 MB


In [18]:
# Round-trip check: reload and confirm it matches.
emb2, meta2 = load_embeddings(EMB_DIR)
assert np.allclose(emb2, embeddings)
assert list(meta2["accession"]) == [str(a) for a in accessions]
print("Checks passed!")


Checks passed!
